# Voxel Allegiance: Visualization & QC

NIfTI map generation, slice overlays, and diagnostic outputs.

**Standalone** — all data loading and computation is included.

## Setup & Configuration

In [1]:
import os
import sys
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '/user_data/csimmon2/git_repos/sym_pt')
from sym_pt_params import (processed_dir, skip_subs, get_sessions,
                           is_patient, get_sub_info, _load_csv)

# ── Category definitions ──────────────────────────────────────────────────────
CATEGORIES = ['face', 'house', 'object', 'word']
CAT_COLORS = {
    'face':   '#E74C3C',
    'house':  '#3498DB',
    'object': '#2ECC71',
    'word':   '#F39C12',
    'none':   '#BDC3C7',
}

RAW_BETA_COPES = {'face': 15, 'house': 16, 'object': 17, 'word': 18}
CAT_VS_ALL_COPES = {'face': 6, 'house': 7, 'object': 8, 'word': 9}
DIFFERENTIAL_COPES = {'face': 1, 'house': 2, 'object': 3, 'word': 4}

ALLEGIANCE_COPE_SET = 'cat_vs_all'
WTA_THRESHOLD = 0

PRE_SURGERY_SESSIONS = {
    'sub-021': ['01'], 'sub-045': ['01'], 'sub-047': ['01'], 'sub-049': ['01'],
    'sub-070': ['01'], 'sub-073': ['01'], 'sub-081': ['01'], 'sub-086': ['01'],
}
EXCLUDE_SUBS = ['OTC108', 'control083', 'control085', 'OTC017', 'control027']

## Helper Functions

In [2]:
def get_cope_map():
    if ALLEGIANCE_COPE_SET == 'raw_beta':
        return RAW_BETA_COPES
    elif ALLEGIANCE_COPE_SET == 'cat_vs_all':
        return CAT_VS_ALL_COPES
    elif ALLEGIANCE_COPE_SET == 'differential':
        return DIFFERENTIAL_COPES
    else:
        raise ValueError(f"Unknown cope set: {ALLEGIANCE_COPE_SET}")


def load_subjects():
    df = _load_csv()
    subjects = {}
    for sub_clean in sorted(df['sub_clean'].unique()):
        if sub_clean in skip_subs:
            continue
        sid = f'sub-{sub_clean}'
        sessions = get_sessions(sub_clean)
        base = os.path.join(processed_dir, sid)
        if not sessions or not os.path.exists(base):
            continue
        info = get_sub_info(sub_clean, sessions[0])
        pt = is_patient(sub_clean)
        intact = info.get('intact_hemi', '')
        code = f"{info.get('group', '')}{sub_clean}"
        if code in EXCLUDE_SUBS:
            continue
        actual_first_ses = f'{sessions[0]:02d}'
        post_sessions = []
        for s in sessions:
            ses_str = f'{s:02d}'
            if sid in PRE_SURGERY_SESSIONS and ses_str in PRE_SURGERY_SESSIONS[sid]:
                continue
            post_sessions.append(ses_str)
        if not post_sessions:
            continue
        subjects[sid] = {
            'code': code, 'sessions': post_sessions,
            'actual_first_ses': actual_first_ses,
            'hemi': ('l' if intact == 'left' else 'r') if pt else None,
            'group': info.get('group', 'unknown'),
            'patient_status': info.get('group', 'unknown'),
            'intact_hemi': intact,
            'surgery_side': ('right' if intact == 'left' else 'left') if pt else 'na',
        }
    return subjects


def get_votc_mask(sub_id, actual_first_ses, hemi):
    roi_dir = os.path.join(processed_dir, sub_id, f'ses-{actual_first_ses}', 'ROIs')
    combined, affine = None, None
    for cat in CATEGORIES:
        mask_path = os.path.join(roi_dir, f'{hemi}_{cat}_searchmask.nii.gz')
        if os.path.exists(mask_path):
            img = nib.load(mask_path)
            data = img.get_fdata() > 0
            if combined is None:
                combined = data.copy(); affine = img.affine
            else:
                combined = combined | data
    if combined is None:
        ventral_path = os.path.join(processed_dir, sub_id, f'ses-{actual_first_ses}',
                                     'derivatives', 'rois', f'{hemi}Ventral.nii.gz')
        if os.path.exists(ventral_path):
            img = nib.load(ventral_path)
            combined = img.get_fdata() > 0; affine = img.affine
    return combined, affine


def load_category_betas(sub_id, session, actual_first_ses, mask, cope_map):
    feat_dir = os.path.join(processed_dir, sub_id, f'ses-{session}',
                            'derivatives', 'fsl', 'loc', 'HighLevel.gfeat')
    if ALLEGIANCE_COPE_SET == 'raw_beta':
        stat_file = 'cope1.nii.gz' if session == actual_first_ses else f'cope1_ses{actual_first_ses}.nii.gz'
    else:
        stat_file = 'zstat1.nii.gz' if session == actual_first_ses else f'zstat1_ses{actual_first_ses}.nii.gz'
    betas = {}
    for cat, cope_num in cope_map.items():
        fpath = os.path.join(feat_dir, f'cope{cope_num}.feat', 'stats', stat_file)
        if not os.path.exists(fpath):
            return None
        betas[cat] = nib.load(fpath).get_fdata()[mask]
    return betas


def compute_wta(betas, threshold=0):
    beta_matrix = np.column_stack([betas[cat] for cat in CATEGORIES])
    max_vals = np.max(beta_matrix, axis=1)
    winners = np.argmax(beta_matrix, axis=1)
    sorted_betas = np.sort(beta_matrix, axis=1)
    margin = sorted_betas[:, -1] - sorted_betas[:, -2]
    if threshold > 0:
        winners[max_vals < threshold] = -1
    return winners, margin, max_vals


def compute_transition_matrix(winners_t1, winners_t2, n_categories=4):
    valid = (winners_t1 >= 0) & (winners_t2 >= 0)
    w1, w2 = winners_t1[valid], winners_t2[valid]
    trans = np.zeros((n_categories, n_categories), dtype=int)
    for i in range(len(w1)):
        trans[w1[i], w2[i]] += 1
    row_sums = trans.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    trans_pct = trans / row_sums * 100
    total_valid = valid.sum()
    retention = np.diag(trans).sum() / total_valid if total_valid > 0 else np.nan
    return trans, trans_pct, retention

## Load Data & Compute WTA + Transitions

In [3]:
# ── Load subjects ─────────────────────────────────────────────────────────────
subjects = load_subjects()
long_subs = {sid: info for sid, info in subjects.items() if len(info['sessions']) >= 2}
cope_map = get_cope_map()
threshold = WTA_THRESHOLD if ALLEGIANCE_COPE_SET == 'raw_beta' else 1.96
print(f"Loaded {len(subjects)} subjects ({len(long_subs)} longitudinal)")

# ── Compute WTA for all subjects, all sessions ───────────────────────────────
# (needed for NIfTI writing)

wta_nifti_dir = os.path.join(processed_dir, 'group_results', 'wta_maps')
os.makedirs(wta_nifti_dir, exist_ok=True)

print("Computing WTA for all subjects...")
wta_count = 0
for sid, info in subjects.items():
    actual_first = info['actual_first_ses']
    hemis = [info['hemi']] if info['patient_status'] == 'OTC' else ['l', 'r']
    for ses in info['sessions']:
        for h in hemis:
            mask, affine = get_votc_mask(sid, actual_first, h)
            if mask is None or int(mask.sum()) == 0:
                continue
            betas = load_category_betas(sid, ses, actual_first, mask, cope_map)
            if betas is None:
                continue
            wta_count += 1

print(f"  {wta_count} subject-session-hemisphere entries to process")

# ── Compute transitions for longitudinal subjects ────────────────────────────

transition_results = {}

for sid, info in long_subs.items():
    sessions = info['sessions']
    t1_ses, t2_ses = sessions[0], sessions[-1]
    actual_first = info['actual_first_ses']
    hemis = [info['hemi']] if info['patient_status'] == 'OTC' else ['l', 'r']

    for h in hemis:
        mask, affine = get_votc_mask(sid, actual_first, h)
        if mask is None:
            continue
        betas_t1 = load_category_betas(sid, t1_ses, actual_first, mask, cope_map)
        betas_t2 = load_category_betas(sid, t2_ses, actual_first, mask, cope_map)
        if betas_t1 is None or betas_t2 is None:
            continue

        winners_t1, margin_t1, _ = compute_wta(betas_t1, threshold)
        winners_t2, margin_t2, _ = compute_wta(betas_t2, threshold)
        trans, trans_pct, retention = compute_transition_matrix(winners_t1, winners_t2)

        transition_results[(sid, h)] = {
            'trans_counts': trans, 'trans_pct': trans_pct, 'retention': retention,
            'winners_t1': winners_t1, 'winners_t2': winners_t2,
            'group': info['patient_status'],
            'surgery_side': info.get('surgery_side', 'na'),
            't1_ses': t1_ses, 't2_ses': t2_ses,
        }

print(f"Computed transitions for {len(transition_results)} subject-hemisphere entries")

Loaded 46 subjects (16 longitudinal)
Computing WTA for all subjects...
  109 subject-session-hemisphere entries to process
Computed transitions for 27 subject-hemisphere entries


## 1. Write WTA Category Maps as NIfTI

In [4]:
# ── Write WTA category maps as NIfTI volumes ─────────────────────────────────
# Values: 1=face, 2=house, 3=object, 4=word, 0=unassigned

for sid, info in subjects.items():
    actual_first = info['actual_first_ses']
    hemis = [info['hemi']] if info['patient_status'] == 'OTC' else ['l', 'r']

    for ses in info['sessions']:
        for h in hemis:
            mask, affine = get_votc_mask(sid, actual_first, h)
            if mask is None:
                continue

            betas = load_category_betas(sid, ses, actual_first, mask, cope_map)
            if betas is None:
                continue

            winners, margin, max_vals = compute_wta(betas, threshold)

            brain_shape = mask.shape
            wta_vol = np.zeros(brain_shape, dtype=np.float32)
            margin_vol = np.zeros(brain_shape, dtype=np.float32)

            voxel_coords = np.where(mask)
            for idx in range(len(winners)):
                cat_val = winners[idx] + 1 if winners[idx] >= 0 else 0
                wta_vol[voxel_coords[0][idx], voxel_coords[1][idx], voxel_coords[2][idx]] = cat_val
                margin_vol[voxel_coords[0][idx], voxel_coords[1][idx], voxel_coords[2][idx]] = margin[idx]

            tag = f'{sid}_ses-{ses}_{h}h'
            nib.save(nib.Nifti1Image(wta_vol, affine),
                     os.path.join(wta_nifti_dir, f'{tag}_wta.nii.gz'))
            nib.save(nib.Nifti1Image(margin_vol, affine),
                     os.path.join(wta_nifti_dir, f'{tag}_margin.nii.gz'))

print(f"Saved WTA NIfTI volumes to: {wta_nifti_dir}")
print("Values: 1=face, 2=house, 3=object, 4=word, 0=unassigned")

Saved WTA NIfTI volumes to: /user_data/csimmon2/sym_pt/group_results/wta_maps
Values: 1=face, 2=house, 3=object, 4=word, 0=unassigned


## 2. Nilearn Ventral View: WTA Maps per Patient

In [ ]:
# ── Nilearn ventral view: WTA maps per patient ──────────────────────────────
try:
    from nilearn import plotting
except ImportError:
    print("nilearn not available — skip this cell")
    plotting = None

wta_cmap = ListedColormap(['#000000', '#E74C3C', '#3498DB', '#2ECC71', '#F39C12'])

if plotting:
    patient_sids = sorted([sid for sid, info in subjects.items()
                           if info['patient_status'] == 'OTC'])

    for sid in patient_sids:
        info = subjects[sid]
        actual_first = info['actual_first_ses']
        h = info['hemi']
        first_ses = info['sessions'][0]

        t1_path = os.path.join(processed_dir, sid, f'ses-{actual_first}', 'anat', 'T1w_brain.nii.gz')
        wta_path = os.path.join(wta_nifti_dir, f'{sid}_ses-{first_ses}_{h}h_wta.nii.gz')

        if not os.path.exists(t1_path) or not os.path.exists(wta_path):
            print(f"  SKIP {sid}: missing files")
            continue

        wta_data = nib.load(wta_path).get_fdata()
        t1_brain = nib.load(t1_path).get_fdata()

        z_counts = [(z, np.sum(wta_data[:, :, z] > 0)) for z in range(wta_data.shape[2])]
        z_counts.sort(key=lambda x: x[1], reverse=True)
        best_slices = sorted([z for z, c in z_counts[:5]])

        surgery = info.get('surgery_side', '?')

        fig, axes = plt.subplots(1, len(best_slices), figsize=(4 * len(best_slices), 4))
        if len(best_slices) == 1:
            axes = [axes]

        for ax, z_slice in zip(axes, best_slices):
            ax.imshow(np.rot90(t1_brain[:, :, z_slice]), cmap='gray', aspect='equal')
            wta_masked = np.ma.masked_where(wta_data[:, :, z_slice] == 0, wta_data[:, :, z_slice])
            ax.imshow(np.rot90(wta_masked), cmap=wta_cmap, vmin=0, vmax=4,
                      aspect='equal', alpha=0.7)
            ax.set_title(f'z={z_slice}', fontsize=9)
            ax.axis('off')

        legend_patches = [Patch(facecolor=c, label=cat)
                          for cat, c in zip(CATEGORIES, ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12'])]
        fig.legend(handles=legend_patches, loc='lower center', ncol=4, fontsize=10)
        plt.suptitle(f'{sid} ({surgery}-resect, {h}H) — WTA Map ({ALLEGIANCE_COPE_SET})', fontsize=12)
        plt.tight_layout(rect=[0, 0.05, 1, 0.95])
        plt.savefig(os.path.join(wta_nifti_dir, f'{sid}_wta_slices.png'), dpi=150, bbox_inches='tight')
        plt.show()

## 3. T1 vs T2 WTA Slice Comparison

In [ ]:
# ── T1 vs T2 WTA slice comparison per patient ────────────────────────────────

wta_cmap = ListedColormap(['#000000', '#E74C3C', '#3498DB', '#2ECC71', '#F39C12'])

for sid in sorted(long_subs.keys()):
    info = long_subs[sid]
    if info['patient_status'] != 'OTC':
        continue

    actual_first = info['actual_first_ses']
    h = info['hemi']
    sessions = info['sessions']
    t1_ses, t2_ses = sessions[0], sessions[-1]
    surgery = info.get('surgery_side', '?')

    t1_path = os.path.join(processed_dir, sid, f'ses-{actual_first}', 'anat', 'T1w_brain.nii.gz')
    wta_t1_path = os.path.join(wta_nifti_dir, f'{sid}_ses-{t1_ses}_{h}h_wta.nii.gz')
    wta_t2_path = os.path.join(wta_nifti_dir, f'{sid}_ses-{t2_ses}_{h}h_wta.nii.gz')

    if not all(os.path.exists(p) for p in [t1_path, wta_t1_path, wta_t2_path]):
        print(f"  SKIP {sid}: missing files")
        continue

    t1_brain = nib.load(t1_path).get_fdata()
    wta_t1_data = nib.load(wta_t1_path).get_fdata()
    wta_t2_data = nib.load(wta_t2_path).get_fdata()

    z_counts = [(z, np.sum(wta_t1_data[:, :, z] > 0)) for z in range(wta_t1_data.shape[2])]
    z_counts.sort(key=lambda x: x[1], reverse=True)
    best_slices = sorted([z for z, c in z_counts[:4]])

    fig, axes = plt.subplots(2, len(best_slices), figsize=(4 * len(best_slices), 7))

    for col, z_slice in enumerate(best_slices):
        for row, (wta_data, label) in enumerate([(wta_t1_data, f'T1 (ses-{t1_ses})'),
                                                   (wta_t2_data, f'T2 (ses-{t2_ses})')]):
            ax = axes[row, col]
            ax.imshow(np.rot90(t1_brain[:, :, z_slice]), cmap='gray', aspect='equal')
            wta_masked = np.ma.masked_where(wta_data[:, :, z_slice] == 0,
                                             wta_data[:, :, z_slice])
            ax.imshow(np.rot90(wta_masked), cmap=wta_cmap, vmin=0, vmax=4,
                      aspect='equal', alpha=0.7)
            ax.set_title(f'{label} z={z_slice}', fontsize=9)
            ax.axis('off')

    legend_patches = [Patch(facecolor=c, label=cat)
                      for cat, c in zip(CATEGORIES, ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12'])]
    fig.legend(handles=legend_patches, loc='lower center', ncol=4, fontsize=10)
    plt.suptitle(f'{sid} ({surgery}-resect, {h}H) — WTA T1 vs T2 ({ALLEGIANCE_COPE_SET})', fontsize=12)
    plt.tight_layout(rect=[0, 0.04, 1, 0.95])
    plt.savefig(os.path.join(wta_nifti_dir, f'{sid}_wta_t1vt2_slices.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 4. Diagnostic: Patient Transition Details

In [7]:
# ── Print detailed patient transition matrices ───────────────────────────────

for (sid, h), res in sorted(transition_results.items()):
    if res['group'] != 'OTC':
        continue

    surgery = res.get('surgery_side', '?')
    tp = res['trans_pct']
    print(f"\n{sid} ({h}H, {surgery}-resection), retention={res['retention']:.3f}")
    print(f"{'':>8} {'→Face':>8} {'→House':>8} {'→Obj':>8} {'→Word':>8}")
    for i, cat in enumerate(CATEGORIES):
        print(f"{cat:>8} {tp[i,0]:>7.1f}% {tp[i,1]:>7.1f}% {tp[i,2]:>7.1f}% {tp[i,3]:>7.1f}%")


sub-004 (lH, right-resection), retention=0.687
            →Face   →House     →Obj    →Word
    face    80.3%     8.2%     5.9%     5.7%
   house     2.0%    73.5%    12.5%    12.1%
  object    11.5%    19.3%    64.2%     5.0%
    word    12.7%     8.7%    33.3%    45.2%

sub-008 (lH, right-resection), retention=0.301
            →Face   →House     →Obj    →Word
    face    42.8%    30.0%    20.2%     7.0%
   house    48.0%    22.3%    27.1%     2.6%
  object    31.8%    48.3%     9.7%    10.2%
    word    34.1%    24.3%    32.4%     9.1%

sub-010 (rH, left-resection), retention=0.482
            →Face   →House     →Obj    →Word
    face    19.5%    20.9%    52.5%     7.0%
   house    19.5%    18.0%    50.6%    11.8%
  object     7.1%    10.7%    77.6%     4.6%
    word     2.7%    27.8%    48.3%    21.2%

sub-021 (rH, left-resection), retention=0.740
            →Face   →House     →Obj    →Word
    face    90.2%     2.9%     6.2%     0.6%
   house     4.4%    90.2%     4.2%     1.3%


## 5. Diagnostic: Control Word→Object Rates

In [8]:
# ── Control word→object transition rate ───────────────────────────────────────

for (sid, h), res in sorted(transition_results.items()):
    if res['group'] == 'control':
        tp = res['trans_pct']
        print(f"{sid} ({h}H): word→obj = {tp[3,2]:.1f}%, word retention = {tp[3,3]:.1f}%")

sub-018 (lH): word→obj = 9.8%, word retention = 70.4%
sub-018 (rH): word→obj = 4.2%, word retention = 23.7%
sub-022 (lH): word→obj = 54.5%, word retention = 25.5%
sub-022 (rH): word→obj = 51.1%, word retention = 11.0%
sub-025 (lH): word→obj = 17.2%, word retention = 62.7%
sub-025 (rH): word→obj = 59.5%, word retention = 22.7%
sub-052 (lH): word→obj = 3.9%, word retention = 84.8%
sub-052 (rH): word→obj = 2.9%, word retention = 84.7%
sub-058 (lH): word→obj = 48.3%, word retention = 49.7%
sub-058 (rH): word→obj = 21.0%, word retention = 75.7%
sub-062 (lH): word→obj = 0.6%, word retention = 86.2%
sub-062 (rH): word→obj = 1.1%, word retention = 93.7%
sub-064 (lH): word→obj = 3.1%, word retention = 94.5%
sub-064 (rH): word→obj = 18.8%, word retention = 74.8%
sub-068 (lH): word→obj = 27.7%, word retention = 43.4%
sub-068 (rH): word→obj = 4.7%, word retention = 76.3%


In [ ]:
# Patient WTA maps
from IPython.display import Image, display
import glob

print("=== PATIENT WTA ===")
pngs = sorted(glob.glob('/user_data/csimmon2/sym_pt/group_results/figures/inflated_wta/sub-*_wta_inflated.png'))
for p in pngs:
    print(p.split('/')[-1])
    display(Image(filename=p, width=600))

In [ ]:
# Group control heatmaps
print("=== GROUP CONTROL HEATMAPS ===")
pngs = sorted(glob.glob('/user_data/csimmon2/sym_pt/group_results/figures/inflated_wta/*heatmap*.png'))
for p in pngs:
    print(p.split('/')[-1])
    display(Image(filename=p, width=800))